# GraphRAG 与 LightRAG：教学示例

这个 Notebook 使用一个很小的企业知识库，演示 GraphRAG 和 LightRAG 的核心数据流。默认只使用 Python 标准库，不需要 LLM API、向量数据库或图数据库。

学习目标：

1. 用实体和关系构建邻接表知识图谱。
2. 实现 GraphRAG 风格的 Local Search。
3. 用连通社区摘要模拟 GraphRAG Global Search。
4. 实现 LightRAG 风格的低层关键词、高层关键词与混合检索。
5. 理解真实框架替换教学组件时需要改动的位置。

> 这是原理教学代码，不是 Microsoft GraphRAG 或 LightRAG 的内部源码复刻。真实项目中的实体关系应由 LLM/NER 抽取，社区发现通常使用 Leiden 等算法，文本检索通常使用 Embedding、BM25 和 Reranker。

## 1. 导入标准库

In [1]:
from __future__ import annotations

from collections import defaultdict, deque
from dataclasses import dataclass
from pprint import pprint
import re
from typing import Any

## 2. 准备小型文档集

数据分成采购和人事两个主题，方便观察局部实体关系与全局社区。

In [2]:
DOCUMENTS = [
    {
        "id": "doc-purchase-1",
        "title": "笔记本采购项目",
        "text": "采购部在 2025 年发起笔记本电脑采购项目，信息部负责技术验收。",
    },
    {
        "id": "doc-purchase-2",
        "title": "供应商参与记录",
        "text": "供应商 A 参与笔记本电脑采购项目，并提交了设备报价。",
    },
    {
        "id": "doc-purchase-3",
        "title": "历史维保服务",
        "text": "供应商 A 曾经为信息部提供服务器维保服务。",
    },
    {
        "id": "doc-purchase-4",
        "title": "预算审核",
        "text": "财务部负责审核笔记本电脑采购项目的预算。",
    },
    {
        "id": "doc-hr-1",
        "title": "请假制度",
        "text": "员工提交请假申请后，由直属上级审批，人事部负责考勤归档。",
    },
    {
        "id": "doc-hr-2",
        "title": "年假规则",
        "text": "年假余额由人事部维护，请假申请会扣减员工的可用年假。",
    },
]

for document in DOCUMENTS:
    print(f"{document['id']}: {document['text']}")

doc-purchase-1: 采购部在 2025 年发起笔记本电脑采购项目，信息部负责技术验收。
doc-purchase-2: 供应商 A 参与笔记本电脑采购项目，并提交了设备报价。
doc-purchase-3: 供应商 A 曾经为信息部提供服务器维保服务。
doc-purchase-4: 财务部负责审核笔记本电脑采购项目的预算。
doc-hr-1: 员工提交请假申请后，由直属上级审批，人事部负责考勤归档。
doc-hr-2: 年假余额由人事部维护，请假申请会扣减员工的可用年假。


## 3. 准备实体关系三元组

真实 GraphRAG/LightRAG 会调用模型，从 chunk 中抽取实体、关系、描述和来源。为了让示例稳定可运行，这里直接准备已抽取好的三元组，并保留 `source_id`。

In [ ]:
@dataclass(frozen=True)
class Relation:
    source: str
    predicate: str
    target: str
    source_id: str


RELATIONS = [
    Relation("采购部", "发起", "笔记本电脑采购项目", "doc-purchase-1"),
    Relation("信息部", "负责技术验收", "笔记本电脑采购项目", "doc-purchase-1"),
    Relation("供应商 A", "参与", "笔记本电脑采购项目", "doc-purchase-2"),
    Relation("供应商 A", "提供", "服务器维保服务", "doc-purchase-3"),
    Relation("服务器维保服务", "服务对象", "信息部", "doc-purchase-3"),
    Relation("财务部", "审核预算", "笔记本电脑采购项目", "doc-purchase-4"),
    Relation("员工", "提交", "请假申请", "doc-hr-1"),
    Relation("直属上级", "审批", "请假申请", "doc-hr-1"),
    Relation("人事部", "归档", "请假申请", "doc-hr-1"),
    Relation("人事部", "维护", "年假余额", "doc-hr-2"),
    Relation("请假申请", "扣减", "年假余额", "doc-hr-2"),
]

pprint(RELATIONS)

## 4. 构建邻接表知识图谱

邻接表同时保存正向和反向入口，但关系本身仍保留原始方向。这样从任意实体出发都能扩展邻居。

In [ ]:
graph: dict[str, list[Relation]] = defaultdict(list)

for relation in RELATIONS:
    graph[relation.source].append(relation)
    graph[relation.target].append(relation)

entities = sorted(graph)
print(f"实体数量: {len(entities)}")
print(f"关系数量: {len(RELATIONS)}")
print("实体列表:")
pprint(entities)

In [ ]:
def other_endpoint(relation: Relation, entity: str) -> str:
    """返回一条关系中与当前实体相对的另一端。"""
    return relation.target if relation.source == entity else relation.source


def relation_text(relation: Relation) -> str:
    return f"{relation.source} --{relation.predicate}--> {relation.target}"


for relation in graph["供应商 A"]:
    print(relation_text(relation), "来源:", relation.source_id)

## 5. 简单文本分词与相似度

为了避免依赖 Embedding，本例使用中文字符、二元词片段和英文单词做一个非常粗糙的 Jaccard 相似度。生产系统应替换成 Dense Embedding + BM25 + Reranker。

In [ ]:
def tokenize(text: str) -> set[str]:
    text = text.lower().replace(" ", "")
    tokens: set[str] = set(re.findall(r"[a-z0-9_]+", text))
    for segment in re.findall(r"[\u4e00-\u9fff]+", text):
        tokens.update(segment)
        tokens.update(segment[index:index + 2] for index in range(len(segment) - 1))
        tokens.update(segment[index:index + 3] for index in range(len(segment) - 2))
    return {token for token in tokens if token}


def jaccard_score(left: str, right: str) -> float:
    left_tokens = tokenize(left)
    right_tokens = tokenize(right)
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)


print(jaccard_score("供应商 A 和信息部有什么联系？", DOCUMENTS[2]["text"]))

## 6. GraphRAG 风格 Local Search

Local Search 的思路是：先在问题中链接实体，再从实体出发扩展一到多跳关系，最后回查支持这些关系的原文。

In [ ]:
def normalize(text: str) -> str:
    return re.sub(r"\s+", "", text).lower()


def match_entities(question: str) -> list[str]:
    normalized_question = normalize(question)
    direct_matches = [
        entity for entity in entities if normalize(entity) in normalized_question
    ]
    if direct_matches:
        return direct_matches

    ranked = sorted(
        ((jaccard_score(question, entity), entity) for entity in entities),
        reverse=True,
    )
    return [entity for score, entity in ranked[:2] if score > 0]


def local_search(question: str, max_depth: int = 2) -> dict[str, Any]:
    seed_entities = match_entities(question)
    queue = deque((entity, 0) for entity in seed_entities)
    visited_entities = set(seed_entities)
    found_relations: list[Relation] = []
    seen_relations: set[Relation] = set()

    while queue:
        entity, depth = queue.popleft()
        for relation in graph.get(entity, []):
            if relation not in seen_relations:
                seen_relations.add(relation)
                found_relations.append(relation)

            neighbor = other_endpoint(relation, entity)
            if depth < max_depth and neighbor not in visited_entities:
                visited_entities.add(neighbor)
                queue.append((neighbor, depth + 1))

    source_ids = {relation.source_id for relation in found_relations}
    source_documents = [
        document for document in DOCUMENTS if document["id"] in source_ids
    ]
    return {
        "question": question,
        "seed_entities": seed_entities,
        "visited_entities": sorted(visited_entities),
        "relations": found_relations,
        "documents": source_documents,
    }

In [ ]:
local_result = local_search("供应商 A 和信息部有什么联系？", max_depth=2)
print("起始实体:", local_result["seed_entities"])
print("扩展实体:", local_result["visited_entities"])
print("关系证据:")
for relation in local_result["relations"]:
    print(" -", relation_text(relation), f"[{relation.source_id}]")
print("原文证据:")
for document in local_result["documents"]:
    print(f" - [{document['id']}] {document['text']}")

从结果可以看到，问题中只明确出现了“供应商 A”和“信息部”，但图遍历能经过“服务器维保服务”找到一条关系路径：

```text
供应商 A -> 提供 -> 服务器维保服务 -> 服务对象 -> 信息部
```

这就是图检索相比单个 chunk 相似度检索更适合多跳关系问题的地方。

## 7. GraphRAG 风格社区与 Global Search

真实 Microsoft GraphRAG 通常使用 Leiden 等算法做层级社区发现，并由 LLM 生成社区报告。本例用连通分量代替社区算法，再用规则拼接摘要。

In [ ]:
def connected_components() -> list[set[str]]:
    remaining = set(entities)
    components: list[set[str]] = []

    while remaining:
        start = next(iter(remaining))
        queue = deque([start])
        component = {start}
        remaining.remove(start)

        while queue:
            entity = queue.popleft()
            for relation in graph[entity]:
                neighbor = other_endpoint(relation, entity)
                if neighbor in remaining:
                    remaining.remove(neighbor)
                    component.add(neighbor)
                    queue.append(neighbor)

        components.append(component)

    return components


COMMUNITIES = connected_components()
for index, community in enumerate(COMMUNITIES, start=1):
    print(f"社区 {index}:", sorted(community))

In [ ]:
def community_topic(nodes: set[str]) -> str:
    joined = "".join(nodes)
    if any(keyword in joined for keyword in ["采购", "供应商", "预算", "验收"]):
        return "采购与供应商管理"
    if any(keyword in joined for keyword in ["请假", "年假", "人事"]):
        return "人事与请假管理"
    return "其他主题"


def build_community_report(nodes: set[str]) -> dict[str, Any]:
    relations = [
        relation
        for relation in RELATIONS
        if relation.source in nodes and relation.target in nodes
    ]
    source_ids = sorted({relation.source_id for relation in relations})
    summary = "；".join(relation_text(relation) for relation in relations)
    return {
        "topic": community_topic(nodes),
        "entities": sorted(nodes),
        "summary": summary,
        "source_ids": source_ids,
    }


COMMUNITY_REPORTS = [build_community_report(nodes) for nodes in COMMUNITIES]
pprint(COMMUNITY_REPORTS)

In [ ]:
GLOBAL_CUES = {"整体", "全局", "主要", "共同", "主题", "风险", "概括", "总结"}


def global_search(question: str, top_k: int = 2) -> list[dict[str, Any]]:
    has_global_cue = any(cue in question for cue in GLOBAL_CUES)
    ranked: list[tuple[float, dict[str, Any]]] = []

    for report in COMMUNITY_REPORTS:
        searchable_text = f"{report['topic']} {report['summary']}"
        score = jaccard_score(question, searchable_text)
        if has_global_cue:
            score += 0.1
        ranked.append((score, report))

    ranked.sort(key=lambda item: item[0], reverse=True)
    return [
        {"score": round(score, 4), **report}
        for score, report in ranked[:top_k]
    ]


for report in global_search("这个知识库包含哪些主要业务主题？"):
    print(f"主题: {report['topic']}，得分: {report['score']}")
    print("摘要:", report["summary"])
    print("来源:", report["source_ids"], "\n")

真实 Global Search 通常会让模型分别阅读多个社区报告，生成局部观点，再把观点汇总成最终答案。这里省略模型调用，只展示被选中的社区和证据。

## 8. LightRAG 风格关键词分层

LightRAG 的关键思想之一是同时使用低层关键词和高层关键词：

- 低层关键词：具体实体、产品、人员、时间。
- 高层关键词：采购管理、供应商合作、预算风险等主题和关系。

In [ ]:
HIGH_LEVEL_RULES = {
    "采购管理": {"采购", "项目", "预算", "报价", "验收"},
    "供应商合作": {"供应商", "服务", "参与", "合作"},
    "人事管理": {"人事", "员工", "考勤", "请假", "年假"},
    "审批流程": {"审批", "申请", "上级", "归档"},
}


def extract_light_keywords(question: str) -> dict[str, list[str]]:
    low_level = match_entities(question)
    high_level = [
        topic
        for topic, keywords in HIGH_LEVEL_RULES.items()
        if any(keyword in question for keyword in keywords)
    ]
    return {
        "low_level": low_level,
        "high_level": high_level,
    }


questions = [
    "供应商 A 和信息部有什么联系？",
    "采购项目涉及哪些部门和供应商？",
    "请假审批和年假余额有什么关系？",
]

for question in questions:
    print(question)
    print(extract_light_keywords(question), "\n")

## 9. LightRAG 风格 Hybrid 检索

下面把三类得分组合起来：

1. `chunk_score`：问题与原始文本的粗略相似度。
2. `entity_score`：关系是否命中低层实体。
3. `topic_score`：关系是否命中高层主题规则。

真实 LightRAG 会使用实体/关系向量、图存储和专门的上下文构建逻辑，本例只模拟融合思想。

In [ ]:
def relation_topic_score(relation: Relation, topics: list[str]) -> float:
    text = relation_text(relation)
    matched = 0
    for topic in topics:
        keywords = HIGH_LEVEL_RULES[topic]
        if any(keyword in text for keyword in keywords):
            matched += 1
    return matched / max(len(topics), 1)


def hybrid_retrieve(question: str, top_k: int = 5) -> dict[str, Any]:
    keywords = extract_light_keywords(question)
    low_level = set(keywords["low_level"])
    high_level = keywords["high_level"]

    relation_rows = []
    for relation in RELATIONS:
        entity_score = float(
            relation.source in low_level or relation.target in low_level
        )
        topic_score = relation_topic_score(relation, high_level)
        source_document = next(
            document for document in DOCUMENTS if document["id"] == relation.source_id
        )
        chunk_score = jaccard_score(question, source_document["text"])
        final_score = 0.45 * entity_score + 0.35 * topic_score + 0.20 * chunk_score
        relation_rows.append(
            {
                "score": round(final_score, 4),
                "relation": relation,
                "document": source_document,
                "entity_score": entity_score,
                "topic_score": round(topic_score, 4),
                "chunk_score": round(chunk_score, 4),
            }
        )

    relation_rows.sort(key=lambda row: row["score"], reverse=True)
    return {
        "question": question,
        "keywords": keywords,
        "results": relation_rows[:top_k],
    }

In [ ]:
hybrid_result = hybrid_retrieve("采购项目涉及哪些部门和供应商？")
print("关键词:", hybrid_result["keywords"])
for row in hybrid_result["results"]:
    print(
        f"{row['score']:.4f}",
        relation_text(row["relation"]),
        f"来源={row['document']['id']}",
        f"entity={row['entity_score']}",
        f"topic={row['topic_score']}",
        f"chunk={row['chunk_score']}",
    )

## 10. 构建可交给 LLM 的证据上下文

生产系统不会把整个图直接塞给模型，而是选择少量高分实体、关系和原文，并明确来源。

In [ ]:
def build_context(question: str, top_k: int = 4) -> str:
    result = hybrid_retrieve(question, top_k=top_k)
    lines = [f"问题：{question}", "", "关系证据："]
    seen_documents: set[str] = set()

    for index, row in enumerate(result["results"], start=1):
        relation = row["relation"]
        document = row["document"]
        lines.append(
            f"R{index}. {relation_text(relation)} [{document['id']}]"
        )
        seen_documents.add(document["id"])

    lines.append("")
    lines.append("原文证据：")
    for document in DOCUMENTS:
        if document["id"] in seen_documents:
            lines.append(f"[{document['id']}] {document['text']}")

    return "\n".join(lines)


context = build_context("供应商 A 和信息部有什么联系？")
print(context)

真实项目可以把上面的 `context` 放入下面的回答 Prompt：

```text
你是企业知识库助手。
只根据关系证据和原文证据回答。
如果证据不足，请明确说明无法确认。
每个关键结论必须标注来源 ID。
```

## 11. 用简单规则生成演示回答

这一节仍然不调用 LLM，只把检索证据整理成可读文本，帮助观察“检索”和“生成”是两个独立阶段。

In [ ]:
def demo_answer(question: str) -> str:
    result = hybrid_retrieve(question, top_k=3)
    if not result["results"] or result["results"][0]["score"] <= 0:
        return "没有找到足够证据，无法回答。"

    evidence_lines = []
    for row in result["results"]:
        relation = row["relation"]
        evidence_lines.append(
            f"- {relation_text(relation)} [{relation.source_id}]"
        )

    return "检索到以下关系证据：\n" + "\n".join(evidence_lines)


print(demo_answer("供应商 A 和信息部有什么联系？"))

## 12. 对比 Local、Global 与 Hybrid

同一个问题不一定适合所有模式。下面用一个小表观察不同问题的推荐入口。

In [ ]:
QUERY_CASES = [
    {
        "question": "供应商 A 和信息部有什么联系？",
        "recommended": "local / hybrid",
        "reason": "问题包含具体实体，并需要多跳关系。",
    },
    {
        "question": "这个知识库有哪些主要业务主题？",
        "recommended": "global",
        "reason": "问题要求跨社区的整体总结。",
    },
    {
        "question": "请假申请会怎样影响年假余额？",
        "recommended": "local / hybrid",
        "reason": "问题围绕具体业务对象和关系。",
    },
    {
        "question": "笔记本采购项目是谁发起的？",
        "recommended": "naive / local",
        "reason": "单跳事实问题不需要昂贵的全局查询。",
    },
]

for case in QUERY_CASES:
    print(f"问题: {case['question']}")
    print(f"推荐: {case['recommended']}")
    print(f"原因: {case['reason']}\n")

## 13. 简单检索评估

这里用期望来源文档做一个极简 Recall@K。真实评估还应加入实体准确率、关系准确率、路径命中率、答案正确率和引用正确率。

In [ ]:
EVALUATION_SET = [
    {
        "question": "供应商 A 和信息部有什么联系？",
        "expected_sources": {"doc-purchase-3"},
    },
    {
        "question": "谁负责审核采购项目预算？",
        "expected_sources": {"doc-purchase-4"},
    },
    {
        "question": "请假申请由谁审批？",
        "expected_sources": {"doc-hr-1"},
    },
]


def retrieved_sources(question: str, top_k: int = 3) -> set[str]:
    result = hybrid_retrieve(question, top_k=top_k)
    return {row["document"]["id"] for row in result["results"]}


def source_recall(expected: set[str], actual: set[str]) -> float:
    return len(expected & actual) / len(expected) if expected else 1.0


recalls = []
for item in EVALUATION_SET:
    actual = retrieved_sources(item["question"])
    recall = source_recall(item["expected_sources"], actual)
    recalls.append(recall)
    print(item["question"])
    print("期望来源:", item["expected_sources"])
    print("检索来源:", actual)
    print("Recall:", recall, "\n")

print("平均来源 Recall:", sum(recalls) / len(recalls))

## 14. 可选：使用 NetworkX 可视化

如果环境已经安装 `networkx` 和 `matplotlib`，下面会画出知识图谱；没有安装时只显示提示，不影响其他示例。

In [ ]:
try:
    import matplotlib.pyplot as plt
    import networkx as nx

    nx_graph = nx.DiGraph()
    for relation in RELATIONS:
        nx_graph.add_edge(
            relation.source,
            relation.target,
            label=relation.predicate,
        )

    plt.figure(figsize=(14, 8))
    positions = nx.spring_layout(nx_graph, seed=7, k=1.2)
    nx.draw_networkx(
        nx_graph,
        positions,
        node_color="#dbeafe",
        edge_color="#64748b",
        node_size=2200,
        font_size=9,
        arrows=True,
    )
    edge_labels = nx.get_edge_attributes(nx_graph, "label")
    nx.draw_networkx_edge_labels(
        nx_graph,
        positions,
        edge_labels=edge_labels,
        font_size=8,
    )
    plt.axis("off")
    plt.title("GraphRAG / LightRAG 教学知识图谱")
    plt.show()
except ImportError:
    print("未安装 networkx 或 matplotlib，跳过可视化。")

## 15. 替换成真实 Microsoft GraphRAG

完成教学示例后，可以创建独立环境体验真实 GraphRAG：

```bash
pip install graphrag
graphrag init --root ./graphrag_workspace
graphrag index --root ./graphrag_workspace
graphrag query --root ./graphrag_workspace --method local --query "供应商 A 和信息部有什么联系？"
```

接入真实框架后，本 Notebook 中的组件大致对应：

| 教学组件 | 真实 GraphRAG 组件 |
|---|---|
| `DOCUMENTS` | documents / text units |
| `RELATIONS` | LLM 抽取后的 entities / relationships |
| 邻接表 | GraphRAG 实体关系图 |
| 连通分量 | Leiden 层级社区 |
| `community_report` | LLM 生成的 community reports |
| `local_search` | Local Search |
| `global_search` | Global Search / Map-Reduce |

> GraphRAG 版本变化较快，请先执行 `graphrag --help` 和 `graphrag query --help`，不要直接复制其他版本的完整 `settings.yaml`。

## 16. 替换成真实 LightRAG

安装：

```bash
pip install lightrag-hku
```

真实调用通常需要提供 LLM 和 Embedding 适配函数：

```python
from lightrag import LightRAG, QueryParam

rag = LightRAG(
    working_dir="./lightrag_workspace",
    llm_model_func=my_llm_complete,
    embedding_func=my_embedding_func,
)

await rag.initialize_storages()
await rag.ainsert("你的文档内容")
answer = await rag.aquery(
    "供应商 A 和信息部有什么联系？",
    param=QueryParam(mode="hybrid"),
)
```

不同版本的初始化、存储和清理 API 可能不同，应以当前 LightRAG README 和示例为准。

## 17. 练习

1. 增加“供应商 B”和新的采购项目，观察实体匹配与图扩展结果。
2. 添加一条连接采购社区和人事社区的关系，观察连通社区如何变化。
3. 把 `jaccard_score` 替换成真实 Embedding 余弦相似度。
4. 增加 BM25，并用 RRF 融合图、Dense 和 Sparse 结果。
5. 为实体增加别名，解决“供应商A”和“A供应商”的实体链接问题。
6. 为文档增加 `tenant_id`，在检索前强制执行租户权限过滤。
7. 为每条关系增加置信度，低置信度关系不进入回答上下文。

## 18. 总结

- GraphRAG Local Search 从实体出发扩展关系和原文，适合具体实体和多跳问题。
- GraphRAG Global Search利用社区报告覆盖整个语料库，适合主题和风险总结。
- LightRAG 通过低层实体、高层主题、图证据和文本证据组合检索。
- 图中的实体和关系是派生数据，最终回答仍要回到原始文档证据。
- 真实项目应重点评估实体消歧、关系准确率、权限传播、增量更新、费用和查询延迟。